<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 그래프 학습 · 추천 시스템 · 02. GCN: 정규화된 이웃 전파

## Semi-Supervised Classification with Graph Convolutional Networks

- **원 논문:** [Semi-Supervised Classification with Graph Convolutional Networks](https://arxiv.org/abs/1609.02907)
- **저자 / 발표:** Thomas N. Kipf, Max Welling · ICLR (2017)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** 선형대수, PyTorch, cross entropy
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** renormalization trick과 2층 GCN을 구현해 일부 label만으로 노드를 분류합니다.

**축소하거나 생략한 부분:** Cora/Citeseer/Pubmed 대신 18개 노드의 homophilic 합성 그래프를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (2) | GCN layer-wise propagation | shape와 유한값 |
| §2.2, Eq. (8) | A+I 및 대칭 degree 정규화 | 대칭성·self-loop |
| §3.1, Eq. (9) | 2층 softmax 모형 | logit shape |
| §3.1, Eq. (10) | labelled subset cross entropy | train/test accuracy와 loss |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** A [N,N], X [N,D] → hidden [N,H] → class logits [N,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$H^{(l+1)}=\sigma(\tilde D^{-1/2}\tilde A\tilde D^{-1/2}H^{(l)}W^{(l)})$$

- **기호 정의:** Ã=A+I, D̃는 Ã의 차수, H는 노드 표현, W는 학습 가능한 선형변환입니다.
- **수식의 역할:** self-loop와 대칭 정규화로 이웃 특징을 안정적으로 섞습니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** Task P의 `preprocess_graph`, 논문 전용 클래스의
  `forward`, `compute_loss`, `training_step`이 §2, Eq. (2)을 구조화합니다.
- **입출력 shape:** 특징 [N, F]와 인접행렬 [N, N] → 클래스 logit [N, C]
- **평가:** labelled subset cross-entropy와 train/test accuracy
- **원문 대비 한계:** Cora/Citeseer/Pubmed 대신 18개 노드의 homophilic 합성 그래프를 사용합니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.2, Eq. (8)
- **이 셀의 책임:** A+I 및 대칭 degree 정규화
- **Tensor shape 계약:** A [N,N], X [N,D] → hidden [N,H] → class logits [N,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 대칭성·self-loop. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def normalize_adjacency(adj: Tensor) -> Tensor:
    a = adj + torch.eye(len(adj), device=adj.device, dtype=adj.dtype)
    inv = a.sum(1).clamp_min(1).pow(-0.5)
    return inv[:, None] * a * inv[None, :]


a_hat = normalize_adjacency(adjacency)
assert torch.allclose(a_hat, a_hat.T) and (torch.diagonal(a_hat) > 0).all()

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (2) / §3.1, Eq. (9)
- **이 셀의 책임:** GCN layer-wise propagation / 2층 softmax 모형
- **Tensor shape 계약:** A [N,N], X [N,D] → hidden [N,H] → class logits [N,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** shape와 유한값 / logit shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_dim, out_dim) * 0.2)

    def forward(self, x: Tensor, a: Tensor) -> Tensor:
        return a @ x @ self.weight


class MiniGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.g1 = GCNLayer(features.shape[1], 12)
        self.g2 = GCNLayer(12, int(labels.max()) + 1)

    def forward(self, x, a):
        return self.g2(F.relu(self.g1(x, a)), a)


model = MiniGCN()
assert model(features, a_hat).shape == (len(features), int(labels.max()) + 1)

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §3.1, Eq. (9) / §3.1, Eq. (10)
- **이 셀의 책임:** 2층 softmax 모형 / labelled subset cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → hidden [N,H] → class logits [N,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** logit shape / train/test accuracy와 loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
model = ACCELERATOR.move(model)
features_device, a_hat_device, labels_device = ACCELERATOR.move(features, a_hat, labels)
train_mask_device, test_mask_device = ACCELERATOR.move(train_mask, test_mask)
optimizer = torch.optim.Adam(model.parameters(), lr=0.04, weight_decay=5e-4)
losses = []
for _ in range(160):
    logits = model(features_device, a_hat_device)
    loss = F.cross_entropy(logits[train_mask_device], labels_device[train_mask_device])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
with torch.no_grad():
    pred = model(features_device, a_hat_device).argmax(1)
    train_acc = float(
        (pred[train_mask_device] == labels_device[train_mask_device]).float().mean()
    )
    test_acc = float(
        (pred[test_mask_device] == labels_device[test_mask_device]).float().mean()
    )
assert losses[-1] < losses[0] and train_acc >= 0.8
plt.plot(losses)
plt.title(f"GCN loss · test acc={test_acc:.2f}")
plt.xlabel("step")
plt.show()
print({"train_accuracy": train_acc, "test_accuracy": test_acc})

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (2) / §2.2, Eq. (8) / §3.1, Eq. (9) / §3.1, Eq. (10)
- **이 셀의 책임:** GCN layer-wise propagation / A+I 및 대칭 degree 정규화 / 2층 softmax 모형 / labelled subset cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → hidden [N,H] → class logits [N,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** shape와 유한값 / 대칭성·self-loop / logit shape / train/test accuracy와 loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 12
    dropout: float = 0.15
    learning_rate: float = 0.03
    steps: int = 18


def preprocess_graph(graph: Tensor) -> Tensor:
    return normalize_adjacency(graph)


class GcnPortfolioModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        class_count: int,
        config: PortfolioConfig,
    ):
        super().__init__()
        self.config = config
        self.input_layer = GCNLayer(input_dim, config.hidden_dim)
        self.output_layer = GCNLayer(config.hidden_dim, class_count)

    def forward(self, inputs: Tensor, graph: Tensor) -> Tensor:
        hidden = F.relu(self.input_layer(inputs, graph))
        hidden = F.dropout(
            hidden,
            p=self.config.dropout,
            training=self.training,
        )
        return self.output_layer(hidden, graph)

    def compute_loss(
        self,
        outputs: Tensor,
        targets: Tensor,
        label_mask: Tensor,
    ) -> Tensor:
        return F.cross_entropy(outputs[label_mask], targets[label_mask])

    def training_step(
        self,
        inputs: Tensor,
        graph: Tensor,
        targets: Tensor,
        label_mask: Tensor,
    ) -> Tensor:
        outputs = self(inputs, graph)
        return self.compute_loss(outputs, targets, label_mask)

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §3.1, Eq. (10)
- **이 셀의 책임:** labelled subset cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → hidden [N,H] → class logits [N,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** train/test accuracy와 loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: GcnPortfolioModel,
    inputs: Tensor,
    graph: Tensor,
    targets: Tensor,
    label_mask: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    model.train()
    for _ in range(model.config.steps):
        loss = model.training_step(inputs, graph, targets, label_mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §3.1, Eq. (10)
- **이 셀의 책임:** labelled subset cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → hidden [N,H] → class logits [N,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** train/test accuracy와 loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: GcnPortfolioModel,
    inputs: Tensor,
    graph: Tensor,
    targets: Tensor,
) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        prediction = model(inputs, graph).argmax(dim=1)
        accuracy = (
            (prediction[test_mask_device] == targets[test_mask_device]).float().mean()
        )
    return {"test_accuracy": float(accuracy.cpu())}


features_device, labels_device = ACCELERATOR.move(features, labels)
train_mask_device, test_mask_device = ACCELERATOR.move(train_mask, test_mask)
portfolio_config = PortfolioConfig()
portfolio_graph = ACCELERATOR.move(preprocess_graph(adjacency))
portfolio_model = ACCELERATOR.move(
    GcnPortfolioModel(
        features.shape[1],
        int(labels.max()) + 1,
        portfolio_config,
    )
)
portfolio_history = fit_portfolio_model(
    portfolio_model,
    features_device,
    portfolio_graph,
    labels_device,
    train_mask_device,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    features_device,
    portfolio_graph,
    labels_device,
)
assert np.isfinite(portfolio_history).all()
assert 0.0 <= portfolio_metrics["test_accuracy"] <= 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> Cora/Citeseer/Pubmed 대신 18개 노드의 homophilic 합성 그래프를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.